<a href="https://colab.research.google.com/github/Odusote-Faruq/MY_CSC309/blob/Week_1/notebooks/CSC309_Week03_Search_BFS_DFS_Student_Centred.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSC309 – Artificial Intelligence  
**Week 3 Lab:** Search & Problem Solving — BFS/DFS on Classic Problems

**Instructor:** Dr Sakinat Folorunso

**Title:** Associate Professor of AI Systems and FAIR Data **Department:** Computer Sciences, Olabisi Onabanjo University, Ago-Iwoye, Ogun State, Nigeria

**Course Code:** CSC 309

**Mode:** Student‑centred, hands‑on in Google Colab

> Every code cell is commented line‑by‑line so you can follow the logic precisely.

## How to use this notebook
1. Start with the **Group Log** and **Do Now**.  
2. Run the **Setup** cell once.  
3. Work through **Tasks**. Edit only cells marked **`# TODO(Student)`**.  
4. Use **Quick Checks** to test your understanding.  
5. Finish with the **Reflection**. If you finish early, try the **Extensions**.

In [1]:
#@title 🧑🏽‍🤝‍🧑🏾 Group Log (fill before you start)
# The '#@param' annotations create form fields in Colab for easy input.

group_members = "Yung, Simzy, Tosin, Faruq"  #@param {type:"string"}  # Names of teammates
roles_notes = "Driver/Navigator, decisions, questions"  #@param {type:"string"}  # Short working notes

print("👥 Group:", group_members)        # Echo the group list for confirmation
print("📝 Notes:", roles_notes)          # Echo the notes so they're preserved in output

👥 Group: Yung, Simzy, Tosin, Faruq
📝 Notes: Driver/Navigator, decisions, questions


### Learning Objectives
- Formulate problems as **state spaces**.  
- Implement **BFS** and **DFS** with path reconstruction.  
- Discuss completeness/optimality and **combinatorial explosion**.

In [2]:
#@title 🔧 Setup (no extra installs)
# BFS/DFS rely only on Python's standard library.

from collections import deque     # 'deque' gives an efficient queue for BFS
print("✅ Setup for Week 3 is complete.")

✅ Setup for Week 3 is complete.


In [3]:
#@title 🧭 BFS and DFS (fully commented)

def bfs(start, is_goal, neighbors):
    """Breadth‑First Search returning the path from start to goal or None."""
    frontier = deque([start])        # Queue of discovered states to expand
    parent = {start: None}           # Dictionary mapping state -> predecessor for path

    while frontier:                  # Continue until queue is empty
        s = frontier.popleft()       # Pop the oldest state (FIFO)
        if is_goal(s):               # If this state is a goal
            path = []                # We'll reconstruct the path by walking parents
            while s is not None:     # Walk backward until the start (None parent)
                path.append(s)       # Add current state to the path
                s = parent[s]        # Move to predecessor
            return list(reversed(path))  # Reverse to get start..goal order

        for n in neighbors(s):       # For each neighbor of the current state
            if n not in parent:      # Skip already‑seen states
                parent[n] = s        # Record how we reached 'n'
                frontier.append(n)   # Push 'n' to the back of the queue
    return None                      # No path found

def dfs(start, is_goal, neighbors, limit=10000):
    """Depth‑First Search with a node expansion limit; returns a path or None."""
    stack = [start]                  # Use a Python list as a LIFO stack
    parent = {start: None}           # Predecessor map for path reconstruction
    steps = 0                        # Counter to avoid infinite exploration

    while stack and steps < limit:   # Stop if stack empty or limit reached
        s = stack.pop()              # Pop the most recent state (LIFO)
        steps += 1                   # Count this expansion
        if is_goal(s):               # If we reached a goal
            path = []                # Reconstruct path exactly like BFS
            while s is not None:
                path.append(s)
                s = parent[s]
            return list(reversed(path))
        for n in neighbors(s):       # Consider neighbors
            if n not in parent:      # Only take unseen states
                parent[n] = s        # Remember predecessor
                stack.append(n)      # Push neighbor to stack
    return None                      # No path found (or limit exceeded)

In [4]:
#@title ⛵ Missionaries & Cannibals (neighbors function, fully commented)

def mc_neighbors(state):
    """Return legal successor states for (M_left, C_left, boat_side)."""
    M, C, side = state                  # Unpack state; side: 0=left, 1=right
    moves = [(1,0),(2,0),(0,1),(0,2),(1,1)]  # Boat can carry these combinations
    result = []                          # Accumulate legal next states here
    for m, c in moves:                   # Try each boat payload
        if side == 0:                    # If boat is on left bank
            M2, C2, side2 = M - m, C - c, 1   # Move people from left to right
        else:                            # If boat is on right bank
            M2, C2, side2 = M + m, C + c, 0   # Move people from right to left
        if 0 <= M2 <= 3 and 0 <= C2 <= 3:      # Keep counts within [0,3]
            # Safety on left: missionaries not outnumbered (unless zero missionaries)
            safe_left = (M2 == 0 or M2 >= C2)
            # Safety on right (computed from complements 3-M2, 3-C2)
            safe_right = ((3 - M2) == 0) or ((3 - M2) >= (3 - C2))
            if safe_left and safe_right:       # Only accept safe states
                result.append((M2, C2, side2)) # Add the legal successor
    return result                              # Return list of legal neighbors

# Quick demo: BFS path length from start to goal
start = (3, 3, 0)                     # All on the left bank, boat on left
goal  = (0, 0, 1)                     # All on the right bank, boat on right
path_bfs = bfs(start, lambda s: s == goal, mc_neighbors)   # Run BFS
print("BFS path length:", len(path_bfs) if path_bfs else None)  # Show result length

BFS path length: 12


### Task — Uniform‑Cost Search (UCS)
Extend BFS into **UCS** using a priority queue; assign unit cost to each boat move.  
Then compare DFS/BFS/UCS by **number of states expanded**.

In [5]:
import heapq
from collections import deque

# --- Uniform-Cost Search (UCS) adapted to your code style ---
def ucs(start, is_goal, neighbors):
    """UCS for Missionaries & Cannibals using unit-cost moves."""
    frontier = [(0, start)]           # Priority queue: (cumulative cost, state)
    parent = {start: None}            # Track path
    cost_so_far = {start: 0}          # Track cumulative cost to reach state
    expanded = 0                      # Count expanded states

    while frontier:
        g, s = heapq.heappop(frontier)  # Pop lowest-cost state
        expanded += 1

        if is_goal(s):                  # Check goal
            path = []
            while s is not None:
                path.append(s)
                s = parent[s]
            return list(reversed(path)), expanded

        for n in neighbors(s):
            new_cost = g + 1            # Each move has cost 1
            if n not in cost_so_far or new_cost < cost_so_far[n]:
                cost_so_far[n] = new_cost
                parent[n] = s
                heapq.heappush(frontier, (new_cost, n))

    return None, expanded  # No solution found


In [6]:
start = (3, 3, 0)
goal = (0, 0, 1)

path_ucs, exp_ucs = ucs(start, lambda s: s == goal, mc_neighbors)

print("UCS path length:", len(path_ucs) if path_ucs else None)
print("UCS expanded states:", exp_ucs)


UCS path length: 12
UCS expanded states: 15


In [7]:
# --- BFS with expanded states counter ---
def bfs_count(start, is_goal, neighbors):
    """BFS that returns (path, expanded_count)."""
    frontier = deque([start])
    parent = {start: None}
    expanded = 0

    while frontier:
        s = frontier.popleft()
        expanded += 1  # Count this state as expanded

        if is_goal(s):
            path = []
            while s is not None:
                path.append(s)
                s = parent[s]
            return list(reversed(path)), expanded

        for n in neighbors(s):
            if n not in parent:
                parent[n] = s
                frontier.append(n)

    return None, expanded


In [8]:
# --- DFS with expanded states counter ---
def dfs_count(start, is_goal, neighbors, limit=10000):
    """DFS that returns (path, expanded_count) and respects a node expansion limit."""
    stack = [start]
    parent = {start: None}
    expanded = 0
    steps = 0

    while stack and steps < limit:
        s = stack.pop()
        steps += 1
        expanded += 1  # Count this state as expanded

        if is_goal(s):
            path = []
            while s is not None:
                path.append(s)
                s = parent[s]
            return list(reversed(path)), expanded

        for n in neighbors(s):
            if n not in parent:
                parent[n] = s
                stack.append(n)

    return None, expanded


In [9]:
start = (3, 3, 0)
goal = (0, 0, 1)

# BFS
path_bfs, exp_bfs = bfs_count(start, lambda s: s == goal, mc_neighbors)
# DFS
path_dfs, exp_dfs = dfs_count(start, lambda s: s == goal, mc_neighbors)
# UCS
path_ucs, exp_ucs = ucs(start, lambda s: s == goal, mc_neighbors)

print("BFS path length:", len(path_bfs) if path_bfs else None, "| Expanded states:", exp_bfs)
print("DFS path length:", len(path_dfs) if path_dfs else None, "| Expanded states:", exp_dfs)
print("UCS path length:", len(path_ucs) if path_ucs else None, "| Expanded states:", exp_ucs)


BFS path length: 12 | Expanded states: 15
DFS path length: 12 | Expanded states: 12
UCS path length: 12 | Expanded states: 15


In [10]:
# --- Run all algorithms (reuse your existing functions) ---
start = (3, 3, 0)
goal = (0, 0, 1)

path_bfs, exp_bfs = bfs_count(start, lambda s: s == goal, mc_neighbors)
path_dfs, exp_dfs = dfs_count(start, lambda s: s == goal, mc_neighbors)
path_ucs, exp_ucs = ucs(start, lambda s: s == goal, mc_neighbors)

# --- Comparison table ---
print("{:<10} {:<12} {:<16} {:<30}".format("Algorithm", "Path Length", "Expanded States", "Notes"))
print("-"*70)
print("{:<10} {:<12} {:<16} {:<30}".format("BFS",
                                           len(path_bfs) if path_bfs else None,
                                           exp_bfs,
                                           "Shortest path guaranteed"))
print("{:<10} {:<12} {:<16} {:<30}".format("DFS",
                                           len(path_dfs) if path_dfs else None,
                                           exp_dfs,
                                           "Not shortest, may expand more"))
print("{:<10} {:<12} {:<16} {:<30}".format("UCS",
                                           len(path_ucs) if path_ucs else None,
                                           exp_ucs,
                                           "Shortest path, cost-based"))


Algorithm  Path Length  Expanded States  Notes                         
----------------------------------------------------------------------
BFS        12           15               Shortest path guaranteed      
DFS        12           12               Not shortest, may expand more 
UCS        12           15               Shortest path, cost-based     
